# CST 407 PYTHON FOR AI
### Project: Real Time Drawing Recognizer Using Google Quick, Draw! Dataset

This project trains a Convolutional Neural Network to classify hand-drawn sketches from the Google Quick, Draw! Dataset.
The model recognizes drawings across **15 categories** and is saved for use in a browser-based interactive demo.

**Deliverables:**
- Trained CNN with ≥ 90% validation accuracy
- Accuracy/loss training curves
- Confusion matrix across all categories
- MobileNetV2 transfer learning comparison
- Failure case visualization
- Saved model for real-time demo

### Load Libraries

In [ ]:
import os
import json
import requests
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    Dropout, BatchNormalization, Input, GlobalAveragePooling2D
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import to_categorical
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")

### 1. Configuration — Categories & Paths

In [ ]:
CATEGORIES = [
    'apple', 'bicycle', 'cat', 'dog', 'fish',
    'house', 'moon', 'mountain', 'star', 'sun',
    'tree', 'airplane', 'car', 'chair', 'flower'
]
NUM_CLASSES         = len(CATEGORIES)
IMG_SIZE            = 28
TRAIN_SAMPLES       = 10000
VAL_SAMPLES         = 2000
DATA_DIR            = 'data'
MODEL_DIR           = 'models'

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Categories ({NUM_CLASSES}): {CATEGORIES}")
print(f"Training samples per class:   {TRAIN_SAMPLES:,}")
print(f"Validation samples per class: {VAL_SAMPLES:,}")

### 2. Download Dataset

Each category is downloaded as a `.npy` file from the Google Quick, Draw! public storage bucket.
Files that already exist are skipped automatically.

In [ ]:
BASE_URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

def download_category(category, data_dir=DATA_DIR):
    path = os.path.join(data_dir, f"{category}.npy")
    if os.path.exists(path):
        print(f"  [{category}] already downloaded — skipping.")
        return path
    url = BASE_URL + f"{category}.npy"
    print(f"  Downloading [{category}]...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with open(path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=65536):
            f.write(chunk)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  [{category}] saved ({size_mb:.1f} MB)")
    return path

print("Downloading Quick, Draw! categories...")
for cat in CATEGORIES:
    download_category(cat)
print("\nAll categories ready.")

### 3. Load & Preprocess Data

Each `.npy` file contains all available drawings for that category stored as `(N, 784)` uint8 arrays.
We load only the first `TRAIN_SAMPLES + VAL_SAMPLES` rows per category using memory-mapped access to avoid loading gigabytes into RAM.

In [ ]:
def load_category(category, n_train, n_val, data_dir=DATA_DIR):
    path  = os.path.join(data_dir, f"{category}.npy")
    data  = np.load(path, mmap_mode='r')
    total = n_train + n_val
    data  = np.array(data[:total])
    return data[:n_train], data[n_train:total]

x_train_parts, x_val_parts = [], []
y_train_parts, y_val_parts = [], []

for label, cat in enumerate(CATEGORIES):
    x_tr, x_vl = load_category(cat, TRAIN_SAMPLES, VAL_SAMPLES)
    x_train_parts.append(x_tr)
    x_val_parts.append(x_vl)
    y_train_parts.append(np.full(len(x_tr), label))
    y_val_parts.append(np.full(len(x_vl), label))
    print(f"  [{cat:12s}]  train={len(x_tr):,}  val={len(x_vl):,}")

x_train_raw = np.concatenate(x_train_parts, axis=0).astype('float32') / 255.0
x_val_raw   = np.concatenate(x_val_parts,   axis=0).astype('float32') / 255.0
y_train_raw = np.concatenate(y_train_parts, axis=0)
y_val_raw   = np.concatenate(y_val_parts,   axis=0)

x_train = x_train_raw.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
x_val   = x_val_raw.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

y_train_cat = to_categorical(y_train_raw, NUM_CLASSES)
y_val_cat   = to_categorical(y_val_raw,   NUM_CLASSES)

shuffle_idx = np.random.permutation(len(x_train))
x_train     = x_train[shuffle_idx]
y_train_cat = y_train_cat[shuffle_idx]

print(f"\nx_train shape: {x_train.shape}")
print(f"x_val shape:   {x_val.shape}")
print(f"y_train shape: {y_train_cat.shape}")
print(f"y_val shape:   {y_val_cat.shape}")

### 4. Visualize Sample Drawings

In [ ]:
y_train_labels = np.argmax(y_train_cat, axis=1)

fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(10, NUM_CLASSES * 1.5))
fig.suptitle('Sample Drawings per Category (Training Set)', fontsize=13)

for i, cat in enumerate(CATEGORIES):
    class_samples = x_train[y_train_labels == i][:5]
    for j in range(5):
        axes[i, j].imshow(class_samples[j].reshape(IMG_SIZE, IMG_SIZE), cmap='gray_r')
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(cat, fontsize=9, rotation=0, labelpad=45, va='center')

plt.tight_layout()
plt.savefig('sample_drawings.png', dpi=100, bbox_inches='tight')
plt.show()

### 5. Primary CNN Model

Architecture from the project proposal:
```
Input (28×28×1)
  → Conv2D (32 filters, 3×3, ReLU) + BatchNorm + MaxPool(2×2)
  → Conv2D (64 filters, 3×3, ReLU) + BatchNorm + MaxPool(2×2)
  → Conv2D (128 filters, 3×3, ReLU) + BatchNorm
  → Flatten → Dense(256, ReLU) → Dropout(0.5)
  → Dense(15, Softmax)
```
BatchNormalization and MaxPooling2D are added as standard practice to stabilize training and control feature map size.

In [ ]:
def build_cnn(num_classes, img_size=IMG_SIZE):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same',
               input_shape=(img_size, img_size, 1)),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),

        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ], name='QuickDraw_CNN')
    return model

cnn_model = build_cnn(NUM_CLASSES)
cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_callbacks = [
    ModelCheckpoint(
        os.path.join(MODEL_DIR, 'cnn_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

history_cnn = cnn_model.fit(
    x_train, y_train_cat,
    epochs=30,
    batch_size=128,
    validation_data=(x_val, y_val_cat),
    callbacks=cnn_callbacks
)

### 6. Training Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_cnn.history['accuracy'],     label='Train')
ax1.plot(history_cnn.history['val_accuracy'], label='Validation')
ax1.set_title('CNN — Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history_cnn.history['loss'],     label='Train')
ax2.plot(history_cnn.history['val_loss'], label='Validation')
ax2.set_title('CNN — Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.suptitle('CNN Training Curves', fontsize=13)
plt.tight_layout()
plt.savefig('cnn_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

### 7. Evaluate on Validation Set

In [ ]:
val_loss, val_acc = cnn_model.evaluate(x_val, y_val_cat, verbose=0)
print(f"CNN Validation Accuracy: {val_acc * 100:.2f}%")
print(f"CNN Validation Loss:     {val_loss:.4f}")
print(f"Google Benchmark:        ~92.00% (345 categories)")
target_met = val_acc >= 0.90
print(f"\nTarget (>= 90%): {'MET' if target_met else 'NOT MET'}")

### 8. Confusion Matrix

In [ ]:
y_pred_probs = cnn_model.predict(x_val, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = np.argmax(y_val_cat,   axis=1)

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CATEGORIES)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title('CNN — Confusion Matrix (Validation Set)', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

### 9. MobileNetV2 Comparison (Stretch Goal)

MobileNetV2 was pretrained on ImageNet with 224×224 RGB images. Quick, Draw! images are 28×28 grayscale.
To adapt the pretrained weights we:
1. Upsample each drawing from 28×28 to 96×96
2. Replicate the single grayscale channel into 3 RGB channels
3. Freeze the base and train only the classification head

In [ ]:
MNV2_SIZE = 96

print(f"Preparing {MNV2_SIZE}x{MNV2_SIZE} RGB data for MobileNetV2...")
x_train_mnv2 = tf.image.resize(
    tf.repeat(x_train, 3, axis=-1), [MNV2_SIZE, MNV2_SIZE]
).numpy()
x_val_mnv2 = tf.image.resize(
    tf.repeat(x_val, 3, axis=-1), [MNV2_SIZE, MNV2_SIZE]
).numpy()
print(f"x_train_mnv2 shape: {x_train_mnv2.shape}")
print(f"x_val_mnv2 shape:   {x_val_mnv2.shape}")

In [ ]:
base_model = MobileNetV2(input_shape=(MNV2_SIZE, MNV2_SIZE, 3),
                         include_top=False, weights='imagenet')
base_model.trainable = False

inputs  = Input(shape=(MNV2_SIZE, MNV2_SIZE, 3))
x       = base_model(inputs, training=False)
x       = GlobalAveragePooling2D()(x)
x       = Dense(256, activation='relu')(x)
x       = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

mobilenet_model = Model(inputs, outputs, name='QuickDraw_MobileNetV2')
mobilenet_model.summary()

In [ ]:
mobilenet_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mv2_callbacks = [
    ModelCheckpoint(
        os.path.join(MODEL_DIR, 'mobilenetv2_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
]

history_mv2 = mobilenet_model.fit(
    x_train_mnv2, y_train_cat,
    epochs=15,
    batch_size=128,
    validation_data=(x_val_mnv2, y_val_cat),
    callbacks=mv2_callbacks
)

In [ ]:
mv2_loss, mv2_acc = mobilenet_model.evaluate(x_val_mnv2, y_val_cat, verbose=0)
print(f"MobileNetV2 Validation Accuracy: {mv2_acc * 100:.2f}%")
print(f"MobileNetV2 Validation Loss:     {mv2_loss:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history_mv2.history['accuracy'],     label='Train')
ax1.plot(history_mv2.history['val_accuracy'], label='Validation')
ax1.set_title('MobileNetV2 — Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history_mv2.history['loss'],     label='Train')
ax2.plot(history_mv2.history['val_loss'], label='Validation')
ax2.set_title('MobileNetV2 — Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.suptitle('MobileNetV2 Training Curves', fontsize=13)
plt.tight_layout()
plt.savefig('mobilenetv2_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

### 10. Save Model for Demo

In [ ]:
cnn_save_path = os.path.join(MODEL_DIR, 'quickdraw_cnn_final.keras')
cnn_model.save(cnn_save_path)
print(f"CNN model saved: {cnn_save_path}")

category_map = {str(i): cat for i, cat in enumerate(CATEGORIES)}
categories_path = os.path.join(MODEL_DIR, 'categories.json')
with open(categories_path, 'w') as f:
    json.dump(category_map, f, indent=2)
print(f"Category map saved: {categories_path}")
print(f"\nCategory map:")
print(json.dumps(category_map, indent=2))

### 11. Failure Case Analysis

Visualizing drawings the model misclassified helps identify which categories are most confusable.

In [ ]:
misclassified = np.where(y_pred != y_true)[0]
print(f"Total misclassified: {len(misclassified):,} / {len(y_true):,}")
print(f"Error rate:          {len(misclassified) / len(y_true) * 100:.2f}%")

per_class_errors = {}
for idx in misclassified:
    true_label = CATEGORIES[y_true[idx]]
    pred_label = CATEGORIES[y_pred[idx]]
    pair = f"{true_label} → {pred_label}"
    per_class_errors[pair] = per_class_errors.get(pair, 0) + 1

print("\nTop 10 most confused pairs (true → predicted):")
for pair, count in sorted(per_class_errors.items(), key=lambda x: -x[1])[:10]:
    print(f"  {pair:30s}  {count}")

In [ ]:
sample_errors = misclassified[:20]
fig, axes = plt.subplots(4, 5, figsize=(14, 12))
axes = axes.flatten()

for i, idx in enumerate(sample_errors):
    axes[i].imshow(x_val[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray_r')
    axes[i].axis('off')
    axes[i].set_title(
        f"True: {CATEGORIES[y_true[idx]]}\nPred: {CATEGORIES[y_pred[idx]]}",
        fontsize=8, color='crimson'
    )

plt.suptitle('Failure Cases — Misclassified Drawings', fontsize=13)
plt.tight_layout()
plt.savefig('failure_cases.png', dpi=100, bbox_inches='tight')
plt.show()

### 12. Analysis Report

In [ ]:
print("=" * 60)
print("ANALYSIS REPORT — Quick, Draw! CNN Classifier")
print("=" * 60)
print(f"\nDataset")
print(f"  Categories:              {NUM_CLASSES}")
print(f"  Training samples total:  {len(x_train):,}")
print(f"  Validation samples total:{len(x_val):,}")
print(f"  Image size:              {IMG_SIZE}x{IMG_SIZE} grayscale")
print(f"\nCNN Model (Primary)")
print(f"  Validation Accuracy:     {val_acc * 100:.2f}%")
print(f"  Validation Loss:         {val_loss:.4f}")
print(f"  Target (>= 90%):         {'MET' if val_acc >= 0.90 else 'NOT MET'}")
print(f"  Google Benchmark:        ~92.00% across 345 categories")
print(f"\nMobileNetV2 (Stretch Goal Comparison)")
print(f"  Validation Accuracy:     {mv2_acc * 100:.2f}%")
print(f"  Validation Loss:         {mv2_loss:.4f}")
print(f"  Input resized to:        {MNV2_SIZE}x{MNV2_SIZE} RGB")
print(f"\nFailure Analysis")
print(f"  Total misclassified:     {len(misclassified):,}")
print(f"  Error rate:              {len(misclassified) / len(y_true) * 100:.2f}%")
print(f"\nSaved Artifacts")
print(f"  CNN model:               {cnn_save_path}")
print(f"  Category map:            {categories_path}")
print(f"  sample_drawings.png")
print(f"  cnn_training_curves.png")
print(f"  confusion_matrix.png")
print(f"  failure_cases.png")
print("=" * 60)